![Faculty logo](../imgs/logo.png)

# Faculty of Engineering Ain Shams University
## Computer and Systems Department
### CSE374s (UG2023) - Machine Learning and Pattern Recognition

## Classification Project

**Presented to:**
- Prof. Hazem Mahmoud Abbass
- Dr. Nesma Mohamed Ibrahim Rezk
- Eng. Hala Mohamed Ahmed Shaheen

**Students:**
- Abdlrhman Hisham Ismail - 2300343
- Mariam Maged Mohammad - 2300670
- Asmaa Salaheldin Abdelhamid - 2300181

---

# Credit Card Default (Taiwan) - Sections 1, 2, and 4

This notebook implements:
1. Problem Framing & Data Understanding
2. Data Acquisition & Documentation
3. Data Cleaning & Preprocessing

Dataset: UCI Default of Credit Card Clients
- Source: https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients

In [1]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
# Reproducible dataset path
DATASET_CANDIDATES = [
    'dataset/default of credit card clients.xls',
    '../dataset/default of credit card clients.xls'
]

df = None
dataset_path = None
for candidate in DATASET_CANDIDATES:
    try:
        df = pd.read_excel(candidate, header=1)
        dataset_path = candidate
        break
    except FileNotFoundError:
        continue

if df is None:
    raise FileNotFoundError('Could not locate dataset file from expected relative paths.')

print(f'Dataset path: {dataset_path}')
print(f'Data shape: {df.shape}')

Dataset path: ../dataset/default of credit card clients.xls
Data shape: (30000, 25)


## 1) Problem Framing & Data Understanding

**Problem being solved**
- Predict whether a credit card client will default on payment in the next month.

**Target variable**
- `default payment next month`
- `0` = non-default, `1` = default

**Classification type**
- Binary classification.

**Expected challenges**
- Class imbalance (default is minority class).
- Financial outliers in bill/payment amounts.
- Mixed feature semantics (some numeric-coded fields are categorical-like).
- Risk of leakage if preprocessing is fit before splitting.

In [3]:
# Quick evidence for data understanding
TARGET_COL = 'default payment next month'

target_counts = df[TARGET_COL].value_counts().sort_index()
imbalance_ratio = target_counts[1] / target_counts.sum()

print('Target counts:')
print(target_counts)
print(f'\nDefault class ratio: {imbalance_ratio:.2%}')

print('\nMissing values (total):', int(df.isna().sum().sum()))
print('Duplicate rows:', int(df.duplicated().sum()))
print('\nData types:')
print(df.dtypes.value_counts())

Target counts:
default payment next month
0    23364
1     6636
Name: count, dtype: int64

Default class ratio: 22.12%

Missing values (total): 0
Duplicate rows: 0

Data types:
int64    25
Name: count, dtype: int64


## 2) Data Acquisition & Documentation

### Dataset source
- UCI ML Repository: https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients

### Planned modifications log
- Drop identifier column (`ID`) because it is not a predictive behavioral feature.
- Keep target labels unchanged (`0/1`).
- Use stratified split to preserve class ratio.
- Keep preprocessing decisions reproducible through explicit notebook cells and fixed random state.

In [4]:
source_url = 'https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients'

modifications_log = [
    'Dropped ID column before modeling.',
    'Preserved original target labels (0=no default, 1=default).',
    'Used stratified train/validation/test split with fixed random_state=42.',
    'Applied preprocessing in a leakage-safe sklearn pipeline (fit only on training data).'
]

print('Source URL:', source_url)
print('Local file:', dataset_path)
print('Raw shape:', df.shape)
print('Modifications:')
for item in modifications_log:
    print('-', item)

Source URL: https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients
Local file: ../dataset/default of credit card clients.xls
Raw shape: (30000, 25)
Modifications:
- Dropped ID column before modeling.
- Preserved original target labels (0=no default, 1=default).
- Used stratified train/validation/test split with fixed random_state=42.
- Applied preprocessing in a leakage-safe sklearn pipeline (fit only on training data).


## 4) Data Cleaning & Preprocessing

### Decisions implemented
- **Missing values**: check first; impute only if needed (median for numerics, mode for categoricals).
- **Categorical encoding**: one-hot encode categorical-like features (`SEX`, `EDUCATION`, `MARRIAGE`, repayment status columns).
- **Scaling**: standardize numeric financial features for scale-sensitive models.
- **Imbalance handling**: keep stratified split and compute class weights for modeling stage.

Note: preprocessing is fit on training data only to avoid test leakage.

In [5]:
# Prepare features/target
work_df = df.copy()
work_df = work_df.drop(columns=['ID'])

X = work_df.drop(columns=[TARGET_COL])
y = work_df[TARGET_COL]

# Stratified split: 70% train, 15% val, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print('Split shapes:')
print('X_train:', X_train.shape, 'y_train:', y_train.shape)
print('X_val:  ', X_val.shape, 'y_val:  ', y_val.shape)
print('X_test: ', X_test.shape, 'y_test: ', y_test.shape)

print('\nClass distribution by split:')
for name, y_part in [('train', y_train), ('val', y_val), ('test', y_test)]:
    ratio = y_part.mean()
    print(f'{name:>5}: default ratio = {ratio:.2%}')

Split shapes:
X_train: (21000, 23) y_train: (21000,)
X_val:   (4500, 23) y_val:   (4500,)
X_test:  (4500, 23) y_test:  (4500,)

Class distribution by split:
train: default ratio = 22.12%
  val: default ratio = 22.11%
 test: default ratio = 22.13%


In [6]:
# Define categorical-like and numeric features
categorical_features = ['SEX', 'EDUCATION', 'MARRIAGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
numeric_features = [col for col in X.columns if col not in categorical_features]

# Build leakage-safe preprocessing pipeline
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print('Processed matrix shapes:')
print('train:', X_train_processed.shape)
print('val:  ', X_val_processed.shape)
print('test: ', X_test_processed.shape)

# Class weights for imbalance-aware modeling (to be used in classifiers later)
neg = int((y_train == 0).sum())
pos = int((y_train == 1).sum())
class_weight_balanced = {
    0: len(y_train) / (2 * neg),
    1: len(y_train) / (2 * pos)
}
print('\nSuggested class_weight for training:', class_weight_balanced)

Processed matrix shapes:
train: (21000, 91)
val:   (4500, 91)
test:  (4500, 91)

Suggested class_weight for training: {0: 0.6420055029043106, 1: 2.2604951560818085}


In [7]:
# Preprocessing summary (kept in notebook only, no files written)
split_summary_rows = [
    ('random_state', 42),
    ('train_ratio', 0.70),
    ('val_ratio', 0.15),
    ('test_ratio', 0.15),
    ('train_default_ratio', float(y_train.mean())),
    ('val_default_ratio', float(y_val.mean())),
    ('test_default_ratio', float(y_test.mean())),
    ('class_weight_0', class_weight_balanced[0]),
    ('class_weight_1', class_weight_balanced[1])
]

split_summary_df = pd.DataFrame(split_summary_rows, columns=['key', 'value'])
split_summary_df

,key,value
0,random_state,42.000000
1,train_ratio,0.700000
2,val_ratio,0.150000
3,test_ratio,0.150000
4,train_default_ratio,0.221190
5,val_default_ratio,0.221111
6,test_default_ratio,0.221333
7,class_weight_0,0.642006
8,class_weight_1,2.260495


## Submission Notes (copy to report)

- **Problem**: predict next-month credit default risk.
- **Target**: `default payment next month` (binary).
- **Main challenge found**: class imbalance (~22% default class).
- **Data quality**: no missing values in this copy; preprocessing still includes robust imputers.
- **Leakage prevention**: split first, then fit preprocessing only on training set.
- **Imbalance handling strategy**: stratified splits and class-weighted training setup.